# OpenEvolve Circle Packing and TSP Tutorial on Colab

This notebook is a self-contained Colab tutorial for running OpenEvolve with a local open-source LLM. It includes two visual problems: circle packing and a Euclidean traveling-salesperson problem.

Recommended Colab setting: `Runtime -> Change runtime type -> T4 GPU` or better.

Colab GPU availability varies over time. If the local LLM cannot start, the score-function verification and fallback visualization cells still work.

References:

- Google Colab FAQ: https://research.google.com/colaboratory/faq.html
- llama-cpp-python OpenAI-compatible server: https://llama-cpp-python.readthedocs.io/en/latest/server/
- Qwen2.5-Coder-7B-Instruct GGUF: https://huggingface.co/bartowski/Qwen2.5-Coder-7B-Instruct-GGUF

## 1. Runtime Check

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    result = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=False)
    print(result.stdout if result.returncode == 0 else result.stderr)
except FileNotFoundError:
    print("nvidia-smi not found. You may be on CPU; local LLM inference will be slow.")

## 2. Install Packages

This cell avoids local CMake builds for `llama-cpp-python`. It chooses CUDA wheels from the actual CUDA runtime libraries available in Colab, not from the driver-only CUDA version shown by `nvidia-smi`, then falls back to a pre-built CPU wheel if needed.


In [ ]:
import ctypes.util
import glob
import importlib.util
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


def step(message):
    print(f"\n{message}", flush=True)


def run(command, *, check=True, env=None):
    print("+ " + " ".join(command), flush=True)
    return subprocess.run(command, check=check, text=True, capture_output=False, env=env)


def run_capture(command):
    result = subprocess.run(command, check=False, text=True, capture_output=True)
    return (result.stdout or "") + (result.stderr or "")


def parse_cuda_version(text):
    if not text:
        return None
    match = re.search(r"(\d+)\.(\d+)", str(text))
    if not match:
        return None
    return int(match.group(1)), int(match.group(2))


def detect_torch_cuda_version():
    try:
        import torch
    except Exception:
        return None
    return parse_cuda_version(getattr(torch.version, "cuda", None))


def detect_nvcc_cuda_version():
    if not shutil.which("nvcc"):
        return None
    output = run_capture(["nvcc", "--version"])
    match = re.search(r"release\s+(\d+)\.(\d+)", output)
    if not match:
        return None
    return int(match.group(1)), int(match.group(2))


def detect_driver_cuda_version():
    if not shutil.which("nvidia-smi"):
        return None
    output = run_capture(["nvidia-smi"])
    match = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", output)
    if not match:
        return None
    return int(match.group(1)), int(match.group(2))


def find_libcudart_paths():
    paths = set()
    patterns = [
        "/usr/local/cuda/lib64/libcudart.so*",
        "/usr/local/cuda/targets/x86_64-linux/lib/libcudart.so*",
        "/usr/local/cuda-*/lib64/libcudart.so*",
        "/usr/local/cuda-*/targets/x86_64-linux/lib/libcudart.so*",
        "/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib/libcudart.so*",
        "/usr/local/lib/python*/site-packages/nvidia/cuda_runtime/lib/libcudart.so*",
    ]
    for pattern in patterns:
        for candidate in glob.glob(pattern):
            if Path(candidate).exists():
                paths.add(str(Path(candidate).resolve()))
    output = run_capture(["ldconfig", "-p"])
    for line in output.splitlines():
        if "libcudart.so" not in line or "=>" not in line:
            continue
        candidate = line.split("=>", 1)[1].strip()
        if Path(candidate).exists():
            paths.add(str(Path(candidate).resolve()))
    found = ctypes.util.find_library("cudart")
    if found:
        paths.add(found)
    return sorted(paths)


def libcudart_major_versions(paths):
    versions = set()
    for path in paths:
        match = re.search(r"libcudart\.so\.(\d+)", path)
        if match:
            versions.add(int(match.group(1)))
    return versions


def prepend_ld_library_path(paths):
    directories = []
    for path in paths:
        path_obj = Path(path)
        if path_obj.is_file():
            directory = str(path_obj.parent)
        elif path_obj.is_dir():
            directory = str(path_obj)
        else:
            continue
        if directory not in directories:
            directories.append(directory)
    if not directories:
        return
    existing = [p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p]
    merged = directories + [p for p in existing if p not in directories]
    os.environ["LD_LIBRARY_PATH"] = ":".join(merged)
    print("CUDA runtime library paths added to LD_LIBRARY_PATH:", directories, flush=True)


SUPPORTED_CUDA_WHEELS = {
    13: [(2, "cu132"), (0, "cu130")],
    12: [(5, "cu125"), (4, "cu124"), (3, "cu123"), (2, "cu122"), (1, "cu121")],
    11: [(8, "cu118")],
}


def select_runtime_cuda_version(torch_cuda, nvcc_cuda, libcudart_majors):
    for version in [torch_cuda, nvcc_cuda]:
        if version and (not libcudart_majors or version[0] in libcudart_majors):
            return version
    supported_majors = sorted(set(libcudart_majors) & set(SUPPORTED_CUDA_WHEELS), reverse=True)
    if supported_majors:
        return supported_majors[0], 99
    return None


def cuda_wheel_candidates(cuda_version):
    if cuda_version is None:
        return []
    major, minor = cuda_version
    return [tag for required_minor, tag in SUPPORTED_CUDA_WHEELS.get(major, []) if minor >= required_minor]


def pip_install_llama_cpp(extra_index_url, *, force_reinstall=False):
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--prefer-binary",
        "--only-binary",
        "llama-cpp-python",
    ]
    if force_reinstall:
        command.append("--force-reinstall")
    command.extend([
        "llama-cpp-python[server]",
        "--extra-index-url",
        extra_index_url,
    ])
    run(command, env=os.environ.copy())


def llama_cpp_package_present():
    return importlib.util.find_spec("llama_cpp") is not None


def llama_cpp_import_status():
    code = r'''
import llama_cpp
support_fn = getattr(llama_cpp, "llama_supports_gpu_offload", None)
if callable(support_fn):
    try:
        print("GPU offload support:", bool(support_fn()))
    except Exception as exc:
        print("GPU offload support: unknown", exc)
else:
    print("GPU offload support: unknown")
print("llama-cpp-python import OK")
'''
    result = subprocess.run(
        [sys.executable, "-c", code],
        check=False,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    output = (result.stdout or "") + (result.stderr or "")
    return result.returncode == 0, output.strip()


def print_tail(label, text, limit=2000):
    text = text or ""
    print(label, flush=True)
    print(text[-limit:], flush=True)


step("[1/4] Upgrading pip tooling...")
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

step("[2/4] Installing base Python packages...")
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "numpy",
    "scipy",
    "matplotlib",
    "pandas",
    "pyyaml",
    "openai",
    "huggingface_hub",
    "requests",
    "ipython",
])

step("[3/4] Base package installation completed")

libcudart_paths = find_libcudart_paths()
libcudart_majors = libcudart_major_versions(libcudart_paths)
prepend_ld_library_path(libcudart_paths)

torch_cuda = detect_torch_cuda_version()
nvcc_cuda = detect_nvcc_cuda_version()
driver_cuda = detect_driver_cuda_version()
runtime_cuda = select_runtime_cuda_version(torch_cuda, nvcc_cuda, libcudart_majors)
cuda_candidates = cuda_wheel_candidates(runtime_cuda)

print("Detected CUDA sources:", flush=True)
print("- torch.version.cuda:", torch_cuda, flush=True)
print("- nvcc runtime:", nvcc_cuda, flush=True)
print("- libcudart major versions:", sorted(libcudart_majors), flush=True)
print("- nvidia-smi driver CUDA:", driver_cuda, "(not used by itself for wheel selection)", flush=True)
print("Selected runtime CUDA version:", runtime_cuda, flush=True)
print("CUDA wheel candidates:", cuda_candidates or "none", flush=True)

package_present = llama_cpp_package_present()
working_before, status_before = llama_cpp_import_status() if package_present else (False, "not installed")
if package_present and not working_before:
    print_tail("Existing llama-cpp-python import failed; reinstalling with a compatible wheel:", status_before)

if working_before and ("GPU offload support: True" in status_before or not cuda_candidates):
    step(f"[4/4] llama-cpp-python already works")
    print(status_before, flush=True)
else:
    installed = False
    force_reinstall = package_present
    if cuda_candidates:
        step("[4/4] Installing pre-built CUDA llama-cpp-python wheel. This should avoid a local CMake build.")
        for tag in cuda_candidates:
            wheel_index = f"https://abetlen.github.io/llama-cpp-python/whl/{tag}"
            print(f"Trying CUDA wheel index: {wheel_index}", flush=True)
            try:
                pip_install_llama_cpp(wheel_index, force_reinstall=force_reinstall)
            except subprocess.CalledProcessError as exc:
                print(f"CUDA wheel install failed for {tag}: exit code {exc.returncode}", flush=True)
                force_reinstall = True
                continue
            ok, status = llama_cpp_import_status()
            if ok:
                print(status, flush=True)
                installed = True
                break
            print_tail(f"CUDA wheel {tag} installed but import failed; trying next candidate:", status)
            force_reinstall = True
    if not installed:
        step("[4/4] Installing pre-built CPU llama-cpp-python wheel. Live OpenEvolve will be slower.")
        pip_install_llama_cpp(
            "https://abetlen.github.io/llama-cpp-python/whl/cpu",
            force_reinstall=True,
        )
        ok, status = llama_cpp_import_status()
        if not ok:
            print_tail("CPU wheel import failed:", status)
            raise RuntimeError("llama-cpp-python installation failed")
        print(status, flush=True)

print("\nPackage installation cell finished.", flush=True)


## 3. Clone and Install OpenEvolve

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path


def run(command, *, cwd=None):
    print("+ " + " ".join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, check=True)


openevolve_repo = Path("/content/openevolve")
if not (openevolve_repo / ".git").is_dir():
    print("Cloning OpenEvolve...", flush=True)
    run([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/algorithmicsuperintelligence/openevolve.git",
        str(openevolve_repo),
    ])
else:
    print(f"OpenEvolve repository already exists: {openevolve_repo}", flush=True)

print("Installing OpenEvolve in editable mode...", flush=True)
run([sys.executable, "-m", "pip", "install", "-e", str(openevolve_repo)])

openevolve = importlib.import_module("openevolve")
print("OpenEvolve import OK:", getattr(openevolve, "__version__", "unknown"), flush=True)


## 4. LLM Settings

These settings are used by the model download, the local OpenAI-compatible server, and OpenEvolve. Run this once before starting the LLM server.


In [ ]:
#@title LLM settings
MODEL_REPO = "bartowski/Qwen2.5-Coder-7B-Instruct-GGUF"  #@param {type:"string"}
MODEL_FILE = "Qwen2.5-Coder-7B-Instruct-Q4_K_M.gguf"  #@param {type:"string"}
MODEL_ALIAS = "local-qwen2.5-coder-7b"  #@param {type:"string"}
LLM_PORT = 8000  #@param {type:"integer"}
LLM_API_BASE = f"http://127.0.0.1:{LLM_PORT}/v1"
N_CTX = 8192  #@param {type:"integer"}
N_GPU_LAYERS = -1  #@param {type:"integer"}

print({
    "MODEL": f"{MODEL_REPO}/{MODEL_FILE}",
    "MODEL_ALIAS": MODEL_ALIAS,
    "LLM_API_BASE": LLM_API_BASE,
    "N_CTX": N_CTX,
    "N_GPU_LAYERS": N_GPU_LAYERS,
})


## 5. Start a Local OpenAI-Compatible LLM

This downloads a Qwen2.5-Coder 7B Q4 GGUF model and starts `llama-cpp-python` as a local server.

If this fails, skip to section 7. You can still inspect the score functions and fallback plots.

In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import requests
from huggingface_hub import hf_hub_download


def llm_models_url():
    return f"http://127.0.0.1:{LLM_PORT}/v1/models"


def llm_server_reachable(timeout=2):
    try:
        response = requests.get(llm_models_url(), timeout=timeout)
        return response.ok, response.json() if response.ok else response.text[-1000:]
    except Exception as exc:
        return False, exc


def stop_llm_server():
    proc = globals().get("LLM_SERVER_PROC")
    if proc is None or not hasattr(proc, "poll") or proc.poll() is not None:
        return
    print(f"Stopping previous LLM server pid={proc.pid}...")
    try:
        if globals().get("LLM_SERVER_PROC_USES_PROCESS_GROUP", False):
            os.killpg(proc.pid, signal.SIGTERM)
        else:
            proc.terminate()
        proc.wait(timeout=10)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        try:
            if globals().get("LLM_SERVER_PROC_USES_PROCESS_GROUP", False):
                os.killpg(proc.pid, signal.SIGKILL)
            else:
                proc.kill()
        except ProcessLookupError:
            pass
        proc.wait(timeout=10)


def start_llm_server(model_path):
    global LLM_SERVER_PROC, LLM_SERVER_PROC_USES_PROCESS_GROUP, LLM_SERVER_LOG_PATH, LLM_SERVER_LOG
    LLM_SERVER_LOG_PATH = Path("/content/llama_cpp_server.log")
    LLM_SERVER_LOG = open(LLM_SERVER_LOG_PATH, "w")
    cmd = [
        sys.executable, "-m", "llama_cpp.server",
        "--model", str(model_path),
        "--host", "127.0.0.1",
        "--port", str(LLM_PORT),
        "--n_ctx", str(N_CTX),
        "--n_gpu_layers", str(N_GPU_LAYERS),
        "--chat_format", "chatml",
    ]
    print("Starting:", " ".join(cmd))
    LLM_SERVER_PROC = subprocess.Popen(
        cmd,
        stdout=LLM_SERVER_LOG,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    LLM_SERVER_PROC_USES_PROCESS_GROUP = True

    for second in range(240):
        if LLM_SERVER_PROC.poll() is not None:
            LLM_SERVER_LOG.flush()
            print(LLM_SERVER_LOG_PATH.read_text(errors="replace")[-4000:])
            raise RuntimeError("LLM server exited early")
        ok, payload = llm_server_reachable(timeout=2)
        if ok:
            print("LLM server ready:", payload)
            return
        if second % 10 == 0:
            print(f"waiting... {second}s")
        time.sleep(1)

    LLM_SERVER_LOG.flush()
    print(LLM_SERVER_LOG_PATH.read_text(errors="replace")[-4000:])
    raise TimeoutError("LLM server did not become ready")


model_dir = Path("/content/models")
model_dir.mkdir(parents=True, exist_ok=True)
print("Downloading model. First download is about 4-5 GB; later runs use the cache.")
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(model_dir),
    local_dir_use_symlinks=False,
)
print("Model path:", model_path)

stop_llm_server()
start_llm_server(model_path)


## 6. LLM Health Check

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=LLM_API_BASE, api_key="not-needed")
response = client.chat.completions.create(
    model=MODEL_ALIAS,
    messages=[{"role": "user", "content": "Return only the word ok."}],
    max_tokens=16,
    temperature=0,
)
print(response.choices[0].message.content)

## 7. Problem Templates and File Generation

The run cell calls `create_problem_files()` using the current parameters, so changing `PROBLEM_TYPE` or problem-specific parameters does not require editing files manually.

In [ ]:
from pathlib import Path
import importlib.util
import os
import textwrap

import pandas as pd

CIRCLE_INITIAL_PROGRAM = r'''
import math
import os
import numpy as np


def packing_size():
    return int(os.environ.get("PACKING_N", "16"))


# EVOLVE-BLOCK-START
CENTER_SPREAD = 0.62
EDGE_MARGIN = 0.02


def initial_centers(n, spread=CENTER_SPREAD):
    """Compressed grid seed; increasing spread visibly moves centers outward."""
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    centers = []
    for k in range(n):
        row = k // cols
        col = k % cols
        x_grid = 0.5 if cols == 1 else col / max(1, cols - 1)
        y_grid = 0.5 if rows == 1 else row / max(1, rows - 1)
        if row % 2 == 1 and cols > 1:
            x_grid += 0.18 / max(1, cols - 1)
        x = 0.5 + spread * (x_grid - 0.5)
        y = 0.5 + spread * (y_grid - 0.5)
        centers.append((
            min(max(x, EDGE_MARGIN), 1.0 - EDGE_MARGIN),
            min(max(y, EDGE_MARGIN), 1.0 - EDGE_MARGIN),
        ))
    return np.asarray(centers, dtype=float)


def compute_max_radii(centers):
    centers = np.asarray(centers, dtype=float)
    n = centers.shape[0]
    radii = np.empty(n, dtype=float)
    for i, (x, y) in enumerate(centers):
        radii[i] = max(0.0, min(x, y, 1.0 - x, 1.0 - y))
    for _ in range(6):
        changed = False
        for i in range(n):
            for j in range(i + 1, n):
                dist = float(np.linalg.norm(centers[i] - centers[j]))
                allowed = max(0.0, dist - 1e-6)
                total = radii[i] + radii[j]
                if total > allowed and total > 0:
                    scale = allowed / total
                    radii[i] *= scale
                    radii[j] *= scale
                    changed = True
        if not changed:
            break
    return radii


def construct_packing():
    n = packing_size()
    centers = initial_centers(n)
    radii = compute_max_radii(centers)
    return centers, radii, float(np.sum(radii))
# EVOLVE-BLOCK-END


def run_packing():
    return construct_packing()
'''

CIRCLE_HACKED_PROGRAM = r'''
# EVOLVE-BLOCK-START
import os
import numpy as np


def construct_packing():
    n = int(os.environ.get("PACKING_N", "16"))
    centers = np.full((n, 2), 0.5, dtype=float)
    radii = np.full(n, 0.5, dtype=float)
    reported_sum = 999.0
    return centers, radii, reported_sum
# EVOLVE-BLOCK-END


def run_packing():
    return construct_packing()
'''

CIRCLE_EVALUATOR = r'''
import os
import pickle
import subprocess
import sys
import tempfile
import time

import numpy as np


REFERENCE_VALUE_N26 = 2.635


def _config():
    return (
        int(os.environ.get("PACKING_N", "26")),
        os.environ.get("SCORE_MODE", "actual_sum_minus_penalty"),
        float(os.environ.get("OVERLAP_WEIGHT", "1.0")),
        float(os.environ.get("BOUNDARY_WEIGHT", "1.0")),
        float(os.environ.get("HONEST_OVERLAP_WEIGHT", "20.0")),
        float(os.environ.get("HONEST_BOUNDARY_WEIGHT", "20.0")),
    )


def validate_packing(centers, radii):
    n = centers.shape[0]
    if np.isnan(centers).any() or np.isnan(radii).any():
        return False
    for i in range(n):
        if radii[i] < 0:
            return False
        x, y = centers[i]
        r = radii[i]
        if x - r < -1e-6 or x + r > 1 + 1e-6 or y - r < -1e-6 or y + r > 1 + 1e-6:
            return False
    for i in range(n):
        for j in range(i + 1, n):
            dist = np.sqrt(np.sum((centers[i] - centers[j]) ** 2))
            if dist < radii[i] + radii[j] - 1e-6:
                return False
    return True


def compute_penalties(centers, radii):
    n = centers.shape[0]
    overlap_penalty = 0.0
    boundary_penalty = 0.0
    if np.isnan(centers).any() or np.isnan(radii).any():
        return 1e6, 1e6
    for i in range(n):
        x, y = centers[i]
        r = radii[i]
        wall_dist = min(x, y, 1 - x, 1 - y)
        boundary_penalty += max(0.0, r - wall_dist)
        boundary_penalty += max(0.0, -r)
    for i in range(n):
        for j in range(i + 1, n):
            dist = np.sqrt(np.sum((centers[i] - centers[j]) ** 2))
            overlap_penalty += max(0.0, (radii[i] + radii[j]) - dist)
    return float(overlap_penalty), float(boundary_penalty)


def run_with_timeout(program_path, timeout_seconds=30):
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False) as temp_file:
        script = f"""
import importlib.util, os, pickle, traceback
import numpy as np
try:
    spec = importlib.util.spec_from_file_location("program", {program_path!r})
    program = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(program)
    centers, radii, sum_radii = program.run_packing()
    with open({temp_file.name!r} + ".results", "wb") as f:
        pickle.dump({{"centers": centers, "radii": radii, "sum_radii": sum_radii}}, f)
except Exception as e:
    traceback.print_exc()
    with open({temp_file.name!r} + ".results", "wb") as f:
        pickle.dump({{"error": str(e)}}, f)
"""
        temp_file.write(script.encode())
        temp_file_path = temp_file.name
    results_path = f"{temp_file_path}.results"
    try:
        process = subprocess.Popen([sys.executable, temp_file_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        try:
            stdout, stderr = process.communicate(timeout=timeout_seconds)
            if process.returncode != 0:
                raise RuntimeError(stderr.decode(errors="replace")[-2000:] or stdout.decode(errors="replace")[-2000:])
            if not os.path.exists(results_path):
                raise RuntimeError("Results file not found")
            with open(results_path, "rb") as f:
                results = pickle.load(f)
            if "error" in results:
                raise RuntimeError(results["error"])
            return results["centers"], results["radii"], results["sum_radii"]
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
            raise TimeoutError(f"Process timed out after {timeout_seconds} seconds")
    finally:
        for cleanup_path in (temp_file_path, results_path):
            try:
                os.unlink(cleanup_path)
            except OSError:
                pass


def _zero_metrics(eval_time, error=None):
    metrics = {
        "combined_score": 0.0,
        "sum_radii": 0.0,
        "reported_sum": 0.0,
        "validity": 0.0,
        "overlap_penalty": 0.0,
        "boundary_penalty": 0.0,
        "center_spread": 0.0,
        "max_center_spread": 0.0,
        "eval_time": float(eval_time),
    }
    if error is not None:
        metrics["error"] = str(error)
    return metrics


def evaluate(program_path):
    n, score_mode, overlap_weight, boundary_weight, honest_overlap_weight, honest_boundary_weight = _config()
    start_time = time.time()
    try:
        centers, radii, reported_sum = run_with_timeout(program_path, timeout_seconds=30)
    except Exception as e:
        return _zero_metrics(time.time() - start_time, error=e)

    centers = np.asarray(centers, dtype=float)
    radii = np.asarray(radii, dtype=float)
    if centers.shape != (n, 2) or radii.shape != (n,):
        metrics = _zero_metrics(time.time() - start_time, error="invalid shapes")
        if score_mode == "intentionally_bad_reported_sum":
            metrics["reported_sum"] = float(reported_sum)
            metrics["combined_score"] = float(reported_sum)
        return metrics

    valid = validate_packing(centers, radii)
    actual_sum = float(np.sum(radii))
    overlap_penalty, boundary_penalty = compute_penalties(centers, radii)
    center_distances = np.linalg.norm(centers - 0.5, axis=1)
    center_spread = float(np.mean(center_distances))
    max_center_spread = float(np.max(center_distances))
    if score_mode == "hard_valid_sum":
        combined_score = actual_sum if valid else 0.0
    elif score_mode == "actual_sum_minus_penalty":
        combined_score = actual_sum - honest_overlap_weight * overlap_penalty - honest_boundary_weight * boundary_penalty
    elif score_mode == "upstream_target_ratio_n26":
        combined_score = (actual_sum / REFERENCE_VALUE_N26) if valid else 0.0
    elif score_mode == "soft_penalty":
        combined_score = float(reported_sum) - overlap_weight * overlap_penalty - boundary_weight * boundary_penalty
    elif score_mode == "intentionally_bad_reported_sum":
        combined_score = float(reported_sum)
    else:
        raise ValueError(f"Unknown SCORE_MODE: {score_mode!r}")

    metrics = {
        "combined_score": float(combined_score),
        "sum_radii": float(actual_sum),
        "reported_sum": float(reported_sum),
        "validity": 1.0 if valid else 0.0,
        "overlap_penalty": float(overlap_penalty),
        "boundary_penalty": float(boundary_penalty),
        "center_spread": center_spread,
        "max_center_spread": max_center_spread,
        "honest_overlap_weight": float(honest_overlap_weight),
        "honest_boundary_weight": float(honest_boundary_weight),
        "eval_time": float(time.time() - start_time),
    }
    if n == 26:
        metrics["reference_ratio"] = float(actual_sum / REFERENCE_VALUE_N26)
        metrics["upstream_target_ratio"] = float((actual_sum / REFERENCE_VALUE_N26) if valid else 0.0)
    print(
        f"Evaluation: mode={score_mode}, valid={valid}, actual_sum={actual_sum:.6f}, "
        f"reported_sum={float(reported_sum):.6f}, combined_score={combined_score:.6f}, "
        f"overlap={overlap_penalty:.4f}, boundary={boundary_penalty:.4f}, center_spread={center_spread:.4f}"
    )
    return metrics
'''

TSP_INITIAL_PROGRAM = r'''# EVOLVE-BLOCK-START
import os
import numpy as np


def generate_cities(n, seed):
    rng = np.random.default_rng(seed)
    return rng.random((n, 2))


def tour_length(cities, tour):
    tour = np.asarray(tour, dtype=int)
    ordered = cities[tour]
    closed = np.vstack([ordered, ordered[:1]])
    return float(np.sum(np.linalg.norm(np.diff(closed, axis=0), axis=1)))


def nearest_neighbor_tour(cities):
    n = len(cities)
    unvisited = set(range(1, n))
    tour = [0]
    current = 0
    while unvisited:
        nxt = min(unvisited, key=lambda j: float(np.linalg.norm(cities[current] - cities[j])))
        tour.append(nxt)
        unvisited.remove(nxt)
        current = nxt
    return np.asarray(tour, dtype=int)


def run_tsp():
    n = int(os.environ.get("TSP_N", "20"))
    seed = int(os.environ.get("TSP_SEED", "0"))
    cities = generate_cities(n, seed)
    tour = nearest_neighbor_tour(cities)
    length = tour_length(cities, tour)
    return cities, tour, length
# EVOLVE-BLOCK-END


def run_packing():
    # Compatibility alias for generic visualizers. OpenEvolve should preserve run_tsp().
    return run_tsp()
'''

TSP_HACKED_PROGRAM = r'''# EVOLVE-BLOCK-START
import os
import numpy as np


def generate_cities(n, seed):
    rng = np.random.default_rng(seed)
    return rng.random((n, 2))


def run_tsp():
    n = int(os.environ.get("TSP_N", "20"))
    seed = int(os.environ.get("TSP_SEED", "0"))
    cities = generate_cities(n, seed)
    tour = np.zeros(n, dtype=int)  # Invalid: repeats city 0 and misses every other city.
    reported_length = 0.0          # Dishonest value for reward-hacking demonstrations.
    return cities, tour, reported_length
# EVOLVE-BLOCK-END


def run_packing():
    return run_tsp()
'''

TSP_EVALUATOR = r'''"""
TSP tutorial evaluator for OpenEvolve.

The city set is fixed by TSP_N and TSP_SEED. Programs may return cities for
visualization, but the evaluator recomputes the canonical city set and scores the
returned tour on that fixed instance.

Score modes (TSP_SCORE_MODE):
- negative_length: honest objective, maximize -actual_tour_length for valid tours.
- soft_penalty: trust reported length but subtract penalties for invalid tours.
- intentionally_bad_reported_length: trust reported length without validation.
"""

import os
import pickle
import subprocess
import sys
import tempfile
import time
import traceback

import numpy as np


def _config():
    n = int(os.environ.get("TSP_N", "20"))
    seed = int(os.environ.get("TSP_SEED", "0"))
    score_mode = os.environ.get("TSP_SCORE_MODE", "negative_length")
    invalid_penalty = float(os.environ.get("TSP_INVALID_PENALTY", "1000.0"))
    return n, seed, score_mode, invalid_penalty


def generate_cities(n, seed):
    rng = np.random.default_rng(seed)
    return rng.random((n, 2))


def tour_length(cities, tour):
    tour = np.asarray(tour, dtype=int)
    ordered = cities[tour]
    closed = np.vstack([ordered, ordered[:1]])
    return float(np.sum(np.linalg.norm(np.diff(closed, axis=0), axis=1)))


def run_with_timeout(program_path, timeout_seconds=30):
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False) as temp_file:
        script = f"""
import importlib.util, os, pickle, traceback
import numpy as np
try:
    spec = importlib.util.spec_from_file_location("program", {program_path!r})
    program = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(program)
    if hasattr(program, "run_tsp"):
        cities, tour, reported_length = program.run_tsp()
    else:
        cities, tour, reported_length = program.run_packing()
    with open({temp_file.name!r} + ".results", "wb") as f:
        pickle.dump({{"cities": cities, "tour": tour, "reported_length": reported_length}}, f)
except Exception as e:
    traceback.print_exc()
    with open({temp_file.name!r} + ".results", "wb") as f:
        pickle.dump({{"error": str(e)}}, f)
"""
        temp_file.write(script.encode())
        temp_file_path = temp_file.name

    results_path = f"{temp_file_path}.results"
    try:
        process = subprocess.Popen([sys.executable, temp_file_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        try:
            stdout, stderr = process.communicate(timeout=timeout_seconds)
            if process.returncode != 0:
                if stderr:
                    print(f"Subprocess stderr: {stderr.decode()}")
                raise RuntimeError(f"Process exited with code {process.returncode}")
            if not os.path.exists(results_path):
                raise RuntimeError("Results file not found")
            with open(results_path, "rb") as f:
                results = pickle.load(f)
            if "error" in results:
                raise RuntimeError(f"Program execution failed: {results['error']}")
            return results["cities"], results["tour"], results["reported_length"]
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
            raise TimeoutError(f"Process timed out after {timeout_seconds} seconds")
    finally:
        for p in (temp_file_path, results_path):
            if os.path.exists(p):
                os.unlink(p)


def _zero_metrics(eval_time, error=None):
    metrics = {
        "combined_score": -1e9,
        "tour_length": 0.0,
        "reported_length": 0.0,
        "validity": 0.0,
        "missing_count": 0.0,
        "duplicate_count": 0.0,
        "out_of_range_count": 0.0,
        "city_mismatch": 0.0,
        "eval_time": float(eval_time),
    }
    if error is not None:
        metrics["error"] = str(error)
    return metrics


def analyze_tour(returned_cities, returned_tour, n, seed):
    canonical_cities = generate_cities(n, seed)
    city_mismatch = 1e6
    city_valid = False
    try:
        returned_cities = np.asarray(returned_cities, dtype=float)
        if returned_cities.shape == canonical_cities.shape:
            city_mismatch = float(np.max(np.abs(returned_cities - canonical_cities)))
            city_valid = bool(np.allclose(returned_cities, canonical_cities, atol=1e-9, rtol=0.0))
    except Exception:
        returned_cities = np.empty((0, 2), dtype=float)

    try:
        raw_tour = np.asarray(returned_tour)
        integer_like = np.all(np.isfinite(raw_tour.astype(float))) and np.allclose(raw_tour.astype(float), np.round(raw_tour.astype(float)))
        tour = raw_tour.astype(int) if integer_like else np.asarray([], dtype=int)
    except Exception:
        tour = np.asarray([], dtype=int)

    length_valid = len(tour) == n
    in_range = length_valid and bool(np.all((0 <= tour) & (tour < n)))
    unique = set(tour.tolist()) if in_range else set()
    missing_count = float(n - len(unique))
    duplicate_count = float(max(0, len(tour) - len(unique)))
    out_of_range_count = 0.0 if in_range else float(max(n, len(tour)))
    valid = bool(city_valid and length_valid and in_range and missing_count == 0 and duplicate_count == 0)
    actual_length = tour_length(canonical_cities, tour) if valid else 0.0
    violation_count = missing_count + duplicate_count + out_of_range_count + (0.0 if city_valid else float(n))
    return canonical_cities, tour, actual_length, valid, violation_count, missing_count, duplicate_count, out_of_range_count, city_mismatch


def evaluate(program_path):
    n, seed, score_mode, invalid_penalty = _config()
    start_time = time.time()
    try:
        returned_cities, returned_tour, reported_length = run_with_timeout(program_path)
    except Exception as e:
        print(f"Evaluation failed to run program: {e}")
        return _zero_metrics(time.time() - start_time, error=e)

    (
        canonical_cities,
        tour,
        actual_length,
        valid,
        violation_count,
        missing_count,
        duplicate_count,
        out_of_range_count,
        city_mismatch,
    ) = analyze_tour(returned_cities, returned_tour, n, seed)
    reported_length = float(reported_length)

    if score_mode == "negative_length":
        combined_score = -actual_length if valid else -1e6 - invalid_penalty * violation_count
    elif score_mode == "soft_penalty":
        combined_score = -reported_length - invalid_penalty * violation_count - city_mismatch
    elif score_mode == "intentionally_bad_reported_length":
        combined_score = -reported_length
    else:
        raise ValueError(f"Unknown TSP_SCORE_MODE: {score_mode!r}")

    eval_time = time.time() - start_time
    metrics = {
        "combined_score": float(combined_score),
        "tour_length": float(actual_length),
        "reported_length": float(reported_length),
        "validity": 1.0 if valid else 0.0,
        "missing_count": float(missing_count),
        "duplicate_count": float(duplicate_count),
        "out_of_range_count": float(out_of_range_count),
        "city_mismatch": float(city_mismatch),
        "eval_time": float(eval_time),
    }
    print(
        f"Evaluation: mode={score_mode}, valid={valid}, length={actual_length:.6f}, "
        f"reported={reported_length:.6f}, score={combined_score:.6f}, violations={violation_count:.1f}, "
        f"city_mismatch={city_mismatch:.3g}, time={eval_time:.2f}s"
    )
    return metrics


if __name__ == "__main__":
    import json

    path = sys.argv[1] if len(sys.argv) > 1 else "initial_program.py"
    print(json.dumps(evaluate(path), indent=2))
'''


NOISO_INITIAL_PROGRAM = r'''# EVOLVE-BLOCK-START
import os
import numpy as np


def construct_subset():
    n = int(os.environ.get("NOISO_N", "8"))
    # Deliberately weak but valid seed: every other point on one row. Because
    # collinear triples are degenerate, they are not counted as triangles.
    points = [(x, 0) for x in range(0, n, 2)]
    return n, np.asarray(points, dtype=int), len(points)
# EVOLVE-BLOCK-END


def run_no_isosceles():
    return construct_subset()


def run_packing():
    return construct_subset()
'''

NOISO_HACKED_PROGRAM = r'''# EVOLVE-BLOCK-START
import os
import numpy as np


def construct_subset():
    n = int(os.environ.get("NOISO_N", "8"))
    points = [(x, y) for x in range(n) for y in range(n)]
    reported_size = 999
    return n, np.asarray(points, dtype=int), reported_size
# EVOLVE-BLOCK-END


def run_no_isosceles():
    return construct_subset()


def run_packing():
    return construct_subset()
'''

NOISO_EVALUATOR = r'''
import itertools
import os
import pickle
import subprocess
import sys
import tempfile
import time
import traceback

import numpy as np


def _config():
    n = int(os.environ.get("NOISO_N", "8"))
    score_mode = os.environ.get("NOISO_SCORE_MODE", "size_minus_penalty")
    iso_penalty = float(os.environ.get("NOISO_ISOSCELES_PENALTY", "1.5"))
    invalid_penalty = float(os.environ.get("NOISO_INVALID_PENALTY", "10.0"))
    return n, score_mode, iso_penalty, invalid_penalty


def run_with_timeout(program_path, timeout_seconds=30):
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False) as temp_file:
        script = f"""
import importlib.util, pickle, traceback
import numpy as np
try:
    spec = importlib.util.spec_from_file_location("program", {program_path!r})
    program = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(program)
    if hasattr(program, "run_no_isosceles"):
        grid_n, selected_points, reported_size = program.run_no_isosceles()
    else:
        grid_n, selected_points, reported_size = program.run_packing()
    with open({temp_file.name!r} + ".results", "wb") as f:
        pickle.dump({{"grid_n": grid_n, "selected_points": selected_points, "reported_size": reported_size}}, f)
except Exception as e:
    traceback.print_exc()
    with open({temp_file.name!r} + ".results", "wb") as f:
        pickle.dump({{"error": str(e)}}, f)
"""
        temp_file.write(script.encode())
        temp_file_path = temp_file.name

    results_path = f"{temp_file_path}.results"
    try:
        process = subprocess.Popen([sys.executable, temp_file_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        try:
            stdout, stderr = process.communicate(timeout=timeout_seconds)
            if process.returncode != 0:
                raise RuntimeError(stderr.decode(errors="replace")[-2000:] or stdout.decode(errors="replace")[-2000:])
            if not os.path.exists(results_path):
                raise RuntimeError("Results file not found")
            with open(results_path, "rb") as f:
                results = pickle.load(f)
            if "error" in results:
                raise RuntimeError(results["error"])
            return results["grid_n"], results["selected_points"], results["reported_size"]
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
            raise TimeoutError(f"Process timed out after {timeout_seconds} seconds")
    finally:
        for cleanup_path in [temp_file_path, results_path]:
            try:
                os.unlink(cleanup_path)
            except OSError:
                pass


def normalize_points(points, n):
    shape_error = 0
    try:
        arr = np.asarray(points, dtype=int)
    except Exception:
        return np.empty((0, 2), dtype=int), 1, 0, 1

    if arr.size == 0:
        arr = np.empty((0, 2), dtype=int)
    if arr.ndim != 2 or arr.shape[1] != 2:
        shape_error = 1
        return np.empty((0, 2), dtype=int), 0, 0, shape_error

    in_bounds = (arr[:, 0] >= 0) & (arr[:, 0] < n) & (arr[:, 1] >= 0) & (arr[:, 1] < n)
    out_of_range_count = int(np.size(in_bounds) - np.count_nonzero(in_bounds))
    in_grid = [tuple(map(int, point)) for point in arr[in_bounds]]
    duplicate_count = len(in_grid) - len(set(in_grid))
    unique_sorted = sorted(set(in_grid))
    return np.asarray(unique_sorted, dtype=int).reshape((-1, 2)), duplicate_count, out_of_range_count, shape_error


def squared_distance(a, b):
    dx = int(a[0]) - int(b[0])
    dy = int(a[1]) - int(b[1])
    return dx * dx + dy * dy


def twice_area(a, b, c):
    return (int(b[0]) - int(a[0])) * (int(c[1]) - int(a[1])) - (int(b[1]) - int(a[1])) * (int(c[0]) - int(a[0]))


def count_isosceles(points):
    count = 0
    examples = []
    for i, j, k in itertools.combinations(range(len(points)), 3):
        a, b, c = points[i], points[j], points[k]
        if twice_area(a, b, c) == 0:
            continue
        d_ab = squared_distance(a, b)
        d_ac = squared_distance(a, c)
        d_bc = squared_distance(b, c)
        if d_ab == d_ac or d_ab == d_bc or d_ac == d_bc:
            count += 1
            if len(examples) < 3:
                examples.append([points[i].tolist(), points[j].tolist(), points[k].tolist()])
    return count, examples


def _zero_metrics(eval_time, error=None):
    metrics = {
        "combined_score": -1e9,
        "subset_size": 0.0,
        "reported_size": 0.0,
        "validity": 0.0,
        "isosceles_count": 0.0,
        "duplicate_count": 0.0,
        "out_of_range_count": 0.0,
        "invalid_point_count": 1.0,
        "eval_time": float(eval_time),
    }
    if error is not None:
        metrics["error"] = str(error)
    return metrics


def evaluate(program_path):
    n, score_mode, iso_penalty, invalid_penalty = _config()
    start_time = time.time()
    try:
        returned_n, points, reported_size = run_with_timeout(program_path, timeout_seconds=30)
    except Exception as e:
        print(f"Evaluation failed to run program: {e}")
        return _zero_metrics(time.time() - start_time, error=e)

    grid_mismatch = 0 if int(returned_n) == n else 1
    selected, duplicate_count, out_of_range_count, shape_error = normalize_points(points, n)
    isosceles_count, examples = count_isosceles(selected)
    invalid_point_count = duplicate_count + out_of_range_count + shape_error + grid_mismatch
    subset_size = len(selected)
    valid = (isosceles_count == 0 and invalid_point_count == 0)

    if score_mode == "hard_valid_size":
        combined_score = float(subset_size if valid else 0.0)
    elif score_mode == "size_minus_penalty":
        combined_score = float(subset_size - iso_penalty * isosceles_count - invalid_penalty * invalid_point_count)
    elif score_mode == "intentionally_bad_reported_size":
        combined_score = float(reported_size)
    else:
        raise ValueError(f"Unknown NOISO_SCORE_MODE: {score_mode!r}")

    eval_time = time.time() - start_time
    metrics = {
        "combined_score": float(combined_score),
        "subset_size": float(subset_size),
        "reported_size": float(reported_size),
        "validity": 1.0 if valid else 0.0,
        "isosceles_count": float(isosceles_count),
        "duplicate_count": float(duplicate_count),
        "out_of_range_count": float(out_of_range_count),
        "invalid_point_count": float(invalid_point_count),
        "grid_n": float(n),
        "eval_time": float(eval_time),
    }
    if examples:
        metrics["example_isosceles_triangle"] = str(examples[0])
    print(
        f"Evaluation: mode={score_mode}, valid={valid}, size={subset_size}, "
        f"reported={reported_size}, isosceles={isosceles_count}, invalid_points={invalid_point_count}, "
        f"score={combined_score:.3f}, time={eval_time:.2f}s"
    )
    return metrics


if __name__ == "__main__":
    import sys
    print(evaluate(sys.argv[1]))
'''

CIRCLE_CONFIG_TEMPLATE = r'''max_iterations: {iterations}
checkpoint_interval: 1
log_level: "INFO"

llm:
  primary_model: "{model_alias}"
  primary_model_weight: 1.0
  api_base: "{api_base}"
  api_key: "not-needed"
  temperature: {temperature}
  top_p: 0.95
  max_tokens: {max_tokens}
  timeout: 600

prompt:
  system_message: |
    You are an expert Python programmer and computational geometry assistant.
    Improve run_packing() for packing {packing_n} circles in a unit square.
    Preserve the run_packing() interface exactly: it must return centers, radii,
    sum_radii. Optimize both centers and radii; changing centers without
    recomputing radii usually will not improve the score. Do not remove the
    imports or packing_size() helper above the EVOLVE-BLOCK. A good strategy is
    to choose centers, compute the largest valid radii from those centers, and
    return the honest sum of those radii. The starter program intentionally
    compresses centers toward the middle; improving CENTER_SPREAD or
    initial_centers should visibly move circles. Valid geometry matters more
    than claiming a large value.
  num_top_programs: 3
  use_template_stochasticity: true

database:
  population_size: 12
  archive_size: 8
  num_islands: 1
  elite_selection_ratio: 0.3
  exploitation_ratio: 0.7

evaluator:
  timeout: 60
  cascade_evaluation: false
  parallel_evaluations: 1
  use_llm_feedback: false

diff_based_evolution: true
allow_full_rewrites: false
'''

TSP_CONFIG_TEMPLATE = r'''max_iterations: {iterations}
checkpoint_interval: 1
log_level: "INFO"

llm:
  primary_model: "{model_alias}"
  primary_model_weight: 1.0
  api_base: "{api_base}"
  api_key: "not-needed"
  temperature: {temperature}
  top_p: 0.95
  max_tokens: {max_tokens}
  timeout: 600

prompt:
  system_message: |
    You are an expert Python programmer and optimization assistant.
    Improve run_tsp() for a fixed Euclidean traveling-salesperson instance with
    {tsp_n} cities generated by TSP_N and TSP_SEED. Preserve the run_tsp()
    interface exactly: it must return cities, tour, reported_length. The tour
    must be a permutation of 0..n-1. Do not change the city generator to make the
    problem easier; the evaluator recomputes the canonical cities independently.
    Minimize the actual closed tour length. Prefer simple deterministic numpy code.
  num_top_programs: 3
  use_template_stochasticity: true

database:
  population_size: 12
  archive_size: 8
  num_islands: 1
  elite_selection_ratio: 0.3
  exploitation_ratio: 0.7

evaluator:
  timeout: 60
  cascade_evaluation: false
  parallel_evaluations: 1
  use_llm_feedback: false

diff_based_evolution: true
allow_full_rewrites: false
'''

NOISO_CONFIG_TEMPLATE = r'''max_iterations: {iterations}
checkpoint_interval: 1
log_level: "INFO"

llm:
  primary_model: "{model_alias}"
  primary_model_weight: 1.0
  api_base: "{api_base}"
  api_key: "not-needed"
  temperature: {temperature}
  top_p: 0.95
  max_tokens: {max_tokens}
  timeout: 600

prompt:
  system_message: |
    You are an expert Python programmer and discrete geometry assistant.
    Improve run_no_isosceles() for the no-isosceles subset problem on a
    {noiso_n} x {noiso_n} integer grid. Preserve the interface exactly: it must
    return grid_n, selected_points, reported_size. selected_points must be a
    list or numpy array of integer coordinate pairs in the grid. Maximize the
    number of selected points while avoiding every nondegenerate isosceles
    triangle among triples of selected points. The evaluator recomputes the
    actual unique in-grid subset and checks all triples, so do not rely on the
    reported_size value. The starter is intentionally weak: every other point on
    one row. A first improvement is selecting more collinear points; stronger
    solutions add carefully chosen off-line points without creating isosceles
    triangles. Prefer simple deterministic code.
  num_top_programs: 3
  use_template_stochasticity: true

database:
  population_size: 12
  archive_size: 8
  num_islands: 1
  elite_selection_ratio: 0.3
  exploitation_ratio: 0.7

evaluator:
  timeout: 60
  cascade_evaluation: false
  parallel_evaluations: 1
  use_llm_feedback: false

diff_based_evolution: true
allow_full_rewrites: false
'''



def active_score_mode():
    if PROBLEM_TYPE == "circle_packing":
        return SCORE_MODE
    if PROBLEM_TYPE == "tsp":
        return TSP_SCORE_MODE
    if PROBLEM_TYPE == "no_isosceles":
        return NOISO_SCORE_MODE
    raise ValueError(f"Unknown PROBLEM_TYPE: {PROBLEM_TYPE!r}")


def problem_environment(problem_type=None):
    problem_type = problem_type or PROBLEM_TYPE
    if problem_type == "circle_packing":
        return {"PACKING_N": str(PACKING_N), "SCORE_MODE": SCORE_MODE}
    if problem_type == "tsp":
        return {"TSP_N": str(TSP_N), "TSP_SEED": str(TSP_SEED), "TSP_SCORE_MODE": TSP_SCORE_MODE}
    if problem_type == "no_isosceles":
        return {"NOISO_N": str(NOISO_N), "NOISO_SCORE_MODE": NOISO_SCORE_MODE}
    raise ValueError(f"Unknown PROBLEM_TYPE: {problem_type!r}")


def problem_label(problem_type=None):
    problem_type = problem_type or PROBLEM_TYPE
    if problem_type == "circle_packing":
        return f"circle_n{PACKING_N}_{SCORE_MODE}"
    if problem_type == "tsp":
        return f"tsp_n{TSP_N}_seed{TSP_SEED}_{TSP_SCORE_MODE}"
    if problem_type == "no_isosceles":
        return f"noiso_n{NOISO_N}_{NOISO_SCORE_MODE}"
    raise ValueError(f"Unknown PROBLEM_TYPE: {problem_type!r}")


def create_problem_files(problem_type=None):
    problem_type = problem_type or PROBLEM_TYPE
    root = Path(WORKDIR)
    root.mkdir(parents=True, exist_ok=True)
    Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)

    if problem_type == "circle_packing":
        problem_dir = root / "circle_packing"
        files = {
            "initial_program.py": CIRCLE_INITIAL_PROGRAM,
            "hacked_program.py": CIRCLE_HACKED_PROGRAM,
            "evaluator.py": CIRCLE_EVALUATOR,
        }
        config_text = CIRCLE_CONFIG_TEMPLATE.format(
            iterations=ITERATIONS,
            model_alias=MODEL_ALIAS,
            api_base=LLM_API_BASE,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            packing_n=PACKING_N,
        )
    elif problem_type == "tsp":
        problem_dir = root / "tsp"
        files = {
            "initial_program.py": TSP_INITIAL_PROGRAM,
            "hacked_program.py": TSP_HACKED_PROGRAM,
            "evaluator.py": TSP_EVALUATOR,
        }
        config_text = TSP_CONFIG_TEMPLATE.format(
            iterations=ITERATIONS,
            model_alias=MODEL_ALIAS,
            api_base=LLM_API_BASE,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            tsp_n=TSP_N,
        )
    elif problem_type == "no_isosceles":
        problem_dir = root / "no_isosceles"
        files = {
            "initial_program.py": NOISO_INITIAL_PROGRAM,
            "hacked_program.py": NOISO_HACKED_PROGRAM,
            "evaluator.py": NOISO_EVALUATOR,
        }
        config_text = NOISO_CONFIG_TEMPLATE.format(
            iterations=ITERATIONS,
            model_alias=MODEL_ALIAS,
            api_base=LLM_API_BASE,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            noiso_n=NOISO_N,
        )
    else:
        raise ValueError(f"Unknown PROBLEM_TYPE: {problem_type!r}")

    problem_dir.mkdir(parents=True, exist_ok=True)
    for name, content in files.items():
        (problem_dir / name).write_text(textwrap.dedent(content).lstrip())
    (problem_dir / "config_tutorial.yaml").write_text(config_text)

    print("Problem type:", problem_type)
    print("Problem directory:", problem_dir)
    print("Wrote files:")
    for path in sorted(problem_dir.iterdir()):
        print("-", path)
    return problem_dir


def load_evaluator(problem_dir):
    evaluator_path = Path(problem_dir) / "evaluator.py"
    spec = importlib.util.spec_from_file_location("tutorial_evaluator", evaluator_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def verify_problem_without_llm(problem_dir, problem_type=None):
    problem_type = problem_type or PROBLEM_TYPE
    env_updates = problem_environment(problem_type)
    old_values = {key: os.environ.get(key) for key in env_updates}
    os.environ.update(env_updates)
    try:
        evaluator = load_evaluator(problem_dir)
        rows = []
        if problem_type == "circle_packing":
            mode_key = "SCORE_MODE"
            modes = [SCORE_MODE]
            metric_keys = ["combined_score", "sum_radii", "reported_sum", "validity", "overlap_penalty", "boundary_penalty", "center_spread", "max_center_spread"]
        elif problem_type == "tsp":
            mode_key = "TSP_SCORE_MODE"
            modes = [TSP_SCORE_MODE]
            metric_keys = ["combined_score", "tour_length", "reported_length", "validity", "missing_count", "duplicate_count", "city_mismatch"]
        elif problem_type == "no_isosceles":
            mode_key = "NOISO_SCORE_MODE"
            modes = [NOISO_SCORE_MODE]
            metric_keys = ["combined_score", "subset_size", "reported_size", "validity", "isosceles_count", "duplicate_count", "out_of_range_count"]
        else:
            raise ValueError(f"Unknown PROBLEM_TYPE: {problem_type!r}")
        for mode in modes:
            os.environ[mode_key] = mode
            for label, filename in [("seed", "initial_program.py"), ("hacked", "hacked_program.py")]:
                metrics = evaluator.evaluate(str(Path(problem_dir) / filename))
                row = {"score_mode": mode, "program": label}
                row.update({key: metrics.get(key) for key in metric_keys})
                rows.append(row)
        display(pd.DataFrame(rows))
    finally:
        for key, value in old_values.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value


## 8. Visualization Helpers


In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display


def checkpoint_number(path):
    match = re.fullmatch(r"checkpoint_(\d+)", path.name)
    return int(match.group(1)) if match else -1


def stable_checkpoints(run_dir, min_age_seconds=1.0):
    checkpoints_dir = Path(run_dir) / "checkpoints"
    if not checkpoints_dir.exists():
        return []
    now = time.time()
    out = []
    for path in checkpoints_dir.glob("checkpoint_*"):
        program_path = path / "best_program.py"
        info_path = path / "best_program_info.json"
        if not program_path.exists() or not info_path.exists():
            continue
        if max(program_path.stat().st_mtime, info_path.stat().st_mtime) > now - min_age_seconds:
            continue
        try:
            info = json.loads(info_path.read_text())
        except Exception:
            continue
        out.append((checkpoint_number(path), path, info))
    return sorted(out, key=lambda x: x[0])


def checkpoint_history(run_dir):
    rows = []
    metric_keys = [
        "combined_score",
        "sum_radii",
        "tour_length",
        "reported_sum",
        "reported_length",
        "validity",
        "overlap_penalty",
        "boundary_penalty",
        "center_spread",
        "max_center_spread",
        "missing_count",
        "duplicate_count",
        "city_mismatch",
        "subset_size",
        "reported_size",
        "isosceles_count",
        "invalid_point_count",
        "out_of_range_count",
    ]
    for number, path, info in stable_checkpoints(run_dir, min_age_seconds=0.0):
        metrics = info.get("metrics", {})
        row = {
            "checkpoint": number,
            "current_iteration": info.get("current_iteration", number),
            "program_birth_iteration": info.get("iteration"),
        }
        row.update({key: metrics.get(key) for key in metric_keys if key in metrics})
        rows.append(row)
    return pd.DataFrame(rows)


def execute_program(program_path, env=None, problem_type="circle_packing", timeout=None):
    if timeout is None:
        timeout = int(globals().get("VISUALIZATION_TIMEOUT", 90))
    if problem_type == "tsp":
        script = f"""
import importlib.util, json, numpy as np
spec = importlib.util.spec_from_file_location('program', {str(program_path)!r})
program = importlib.util.module_from_spec(spec)
spec.loader.exec_module(program)
if hasattr(program, 'run_tsp'):
    cities, tour, reported_length = program.run_tsp()
else:
    cities, tour, reported_length = program.run_packing()
print(json.dumps({{
    'cities': np.asarray(cities, dtype=float).tolist(),
    'tour': np.asarray(tour, dtype=int).tolist(),
    'reported_length': float(reported_length),
}}))
"""
    elif problem_type == "no_isosceles":
        script = f"""
import importlib.util, json, numpy as np
spec = importlib.util.spec_from_file_location('program', {str(program_path)!r})
program = importlib.util.module_from_spec(spec)
spec.loader.exec_module(program)
if hasattr(program, 'run_no_isosceles'):
    grid_n, selected_points, reported_size = program.run_no_isosceles()
else:
    grid_n, selected_points, reported_size = program.run_packing()
print(json.dumps({{
    'grid_n': int(grid_n),
    'selected_points': np.asarray(selected_points, dtype=int).tolist(),
    'reported_size': float(reported_size),
}}))
"""
    else:
        script = f"""
import importlib.util, json, numpy as np
spec = importlib.util.spec_from_file_location('program', {str(program_path)!r})
program = importlib.util.module_from_spec(spec)
spec.loader.exec_module(program)
centers, radii, sum_radii = program.run_packing()
print(json.dumps({{
    'centers': np.asarray(centers, dtype=float).tolist(),
    'radii': np.asarray(radii, dtype=float).tolist(),
    'sum_radii': float(sum_radii),
}}))
"""
    try:
        result = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, timeout=timeout, env=env)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            f"best_program.py did not finish within VISUALIZATION_TIMEOUT={timeout}s. "
            "Increase VISUALIZATION_TIMEOUT in the Parameters cell if the evolved program uses scipy or another slow optimizer."
        ) from exc
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-2000:])
    return json.loads(result.stdout)


def plot_packing(centers, radii, title):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_aspect("equal")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, linewidth=2))
    for (x, y), r in zip(centers, radii):
        ax.add_patch(plt.Circle((x, y), r, fill=False, linewidth=1.5))
        ax.plot([x], [y], marker=".", markersize=3)
    ax.set_title(title)
    plt.show()


def plot_tsp(cities, tour, title):
    cities = np.asarray(cities, dtype=float)
    tour = np.asarray(tour, dtype=int)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(cities[:, 0], cities[:, 1], s=24)
    for idx, (x, y) in enumerate(cities):
        ax.text(x, y, str(idx), fontsize=7, ha="center", va="bottom")
    if len(tour) == len(cities) and np.all((0 <= tour) & (tour < len(cities))):
        ordered = cities[tour]
        closed = np.vstack([ordered, ordered[:1]])
        ax.plot(closed[:, 0], closed[:, 1], linewidth=1.5)
    else:
        ax.set_title(title + " (invalid tour shown as points only)")
        plt.show()
        return
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    plt.show()


def plot_no_isosceles(grid_n, selected_points, title):
    selected_points = np.asarray(selected_points, dtype=int)
    fig, ax = plt.subplots(figsize=(5, 5))
    all_points = np.array([(x, y) for x in range(grid_n) for y in range(grid_n)], dtype=int)
    if len(all_points):
        ax.scatter(all_points[:, 0], all_points[:, 1], s=18, color="#dddddd", label="grid")
    if selected_points.size:
        selected_points = selected_points.reshape((-1, 2))
        ax.scatter(selected_points[:, 0], selected_points[:, 1], s=48, color="#1f77b4", label="selected")
        for idx, (x, y) in enumerate(selected_points):
            ax.text(x, y + 0.08, str(idx), fontsize=7, ha="center", va="bottom")
    ax.set_xlim(-0.5, grid_n - 0.5)
    ax.set_ylim(-0.5, grid_n - 0.5)
    ax.set_xticks(range(grid_n))
    ax.set_yticks(range(grid_n))
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_title(title + f" (size={len(selected_points) if selected_points.size else 0})")
    ax.legend(loc="upper right")
    plt.show()


def plot_solution(data, problem_type, title):
    if problem_type == "tsp":
        plot_tsp(np.asarray(data["cities"]), np.asarray(data["tour"]), title)
    elif problem_type == "no_isosceles":
        plot_no_isosceles(int(data["grid_n"]), np.asarray(data["selected_points"]), title)
    else:
        plot_packing(np.asarray(data["centers"]), np.asarray(data["radii"]), title)


def show_run_state(run_dir, env=None, problem_type="circle_packing"):
    hist = checkpoint_history(run_dir)
    stable = stable_checkpoints(run_dir)
    if hist.empty:
        checkpoints_dir = Path(run_dir) / "checkpoints"
        if checkpoints_dir.exists():
            checkpoint_names = sorted(path.name for path in checkpoints_dir.glob("checkpoint_*"))
            print("Checkpoint directory exists, but no readable checkpoint metadata is available yet.")
            print("Seen checkpoint directories:", checkpoint_names[-5:] or "none")
        else:
            print("No checkpoint directory yet. Waiting for OpenEvolve to finish its first evaluation.")
        return

    display(hist.tail(10))
    if "combined_score" in hist and hist["combined_score"].notna().any():
        score_columns = ["combined_score"]
    elif problem_type == "circle_packing" and "sum_radii" in hist and hist["sum_radii"].notna().any():
        score_columns = ["sum_radii"]
    elif problem_type == "tsp" and "tour_length" in hist and hist["tour_length"].notna().any():
        score_columns = ["tour_length"]
    elif problem_type == "no_isosceles" and "subset_size" in hist and hist["subset_size"].notna().any():
        score_columns = ["subset_size"]
    else:
        score_columns = []

    if score_columns:
        column = score_columns[0]
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(hist["current_iteration"], hist[column], marker="o", label=column)
        ax.set_xlabel("current_iteration")
        ax.set_ylabel(column)
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()
    else:
        print("Checkpoint metadata is readable, but score metrics are not available yet.")

    if not stable:
        print("Checkpoint metadata is visible, but no stable checkpoint is ready for plotting yet.")
        print("Keeping the watcher alive; the next refresh should render once files stop changing.")
        return

    number, checkpoint_dir, info = stable[-1]
    print("Latest stable checkpoint:", checkpoint_dir.name)
    print("Metrics:", info.get("metrics", {}))
    try:
        data = execute_program(checkpoint_dir / "best_program.py", env=env, problem_type=problem_type)
        plot_solution(data, problem_type, f"Best {problem_type} at checkpoint {number}")
    except Exception as exc:
        print("Could not execute best_program.py:", exc)


## 9. Parameters

Change parameters here, then run the next cell. Setup cells above this point usually only need to run once per Colab runtime.

In [ ]:
#@title Tutorial parameters
PROBLEM_TYPE = "circle_packing"  #@param ["circle_packing", "tsp", "no_isosceles"]

PACKING_N = "10"  #@param ["10", "16", "26"]
PACKING_N = int(PACKING_N)
SCORE_MODE = "actual_sum_minus_penalty"  #@param ["actual_sum_minus_penalty", "hard_valid_sum", "upstream_target_ratio_n26", "soft_penalty", "intentionally_bad_reported_sum"]

TSP_N = "20"  #@param ["12", "20", "30"]
TSP_N = int(TSP_N)
TSP_SEED = 0  #@param {type:"integer"}
TSP_SCORE_MODE = "negative_length"  #@param ["negative_length", "soft_penalty", "intentionally_bad_reported_length"]

NOISO_N = "8"  #@param ["6", "8", "10"]
NOISO_N = int(NOISO_N)
NOISO_SCORE_MODE = "size_minus_penalty"  #@param ["size_minus_penalty", "hard_valid_size", "intentionally_bad_reported_size"]

ITERATIONS = 30  #@param {type:"integer"}
TEMPERATURE = 0.7  #@param {type:"slider", min:0.1, max:1.2, step:0.1}
MAX_TOKENS = "2048"  #@param ["1024", "1536", "2048"]
MAX_TOKENS = int(MAX_TOKENS)
VISUALIZATION_TIMEOUT = 90  #@param {type:"integer"}

WORKDIR = "/content/openevolve_colab_tutorial"
RUN_ROOT = "/content/openevolve_runs"

print({
    "PROBLEM_TYPE": PROBLEM_TYPE,
    "PACKING_N": PACKING_N if PROBLEM_TYPE == "circle_packing" else None,
    "SCORE_MODE": SCORE_MODE if PROBLEM_TYPE == "circle_packing" else None,
    "TSP_N": TSP_N if PROBLEM_TYPE == "tsp" else None,
    "TSP_SEED": TSP_SEED if PROBLEM_TYPE == "tsp" else None,
    "TSP_SCORE_MODE": TSP_SCORE_MODE if PROBLEM_TYPE == "tsp" else None,
    "NOISO_N": NOISO_N if PROBLEM_TYPE == "no_isosceles" else None,
    "NOISO_SCORE_MODE": NOISO_SCORE_MODE if PROBLEM_TYPE == "no_isosceles" else None,
    "ITERATIONS": ITERATIONS,
    "TEMPERATURE": TEMPERATURE,
    "MAX_TOKENS": MAX_TOKENS,
    "VISUALIZATION_TIMEOUT": VISUALIZATION_TIMEOUT,
    "MODEL": f"{MODEL_REPO}/{MODEL_FILE}",
})


## 10. Run OpenEvolve and Watch Checkpoints


In [ ]:
import datetime as dt
import os
import queue
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

import requests
from huggingface_hub import hf_hub_download


def _terminate_open_evolve_process(proc, label="OpenEvolve", use_process_group=False):
    if proc is None or proc.poll() is not None:
        return
    print(f"Stopping previous {label} process pid={proc.pid}...")
    try:
        if use_process_group:
            os.killpg(proc.pid, signal.SIGTERM)
        else:
            proc.terminate()
        proc.wait(timeout=10)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        try:
            if use_process_group:
                os.killpg(proc.pid, signal.SIGKILL)
            else:
                proc.kill()
        except ProcessLookupError:
            pass
        proc.wait(timeout=10)


# If the previous run cell was interrupted, its child openevolve-run.py process
# can keep calling the LLM in the background. Stop it before launching a new run.
for _name in ("OPENEVOLVE_RUN_PROC", "proc"):
    _previous_proc = globals().get(_name)
    if _previous_proc is not None and hasattr(_previous_proc, "poll"):
        _terminate_open_evolve_process(
            _previous_proc,
            label=f"stale {_name}",
            use_process_group=bool(globals().get(f"{_name}_USES_PROCESS_GROUP", False)),
        )

def llm_models_url():
    return f"http://127.0.0.1:{LLM_PORT}/v1/models"


def llm_server_reachable(timeout=2):
    try:
        response = requests.get(llm_models_url(), timeout=timeout)
        return response.ok, response.json() if response.ok else response.text[-1000:]
    except Exception as exc:
        return False, exc


def stop_llm_server():
    proc = globals().get("LLM_SERVER_PROC")
    if proc is None or not hasattr(proc, "poll") or proc.poll() is not None:
        return
    print(f"Stopping stale LLM server pid={proc.pid}...")
    try:
        if globals().get("LLM_SERVER_PROC_USES_PROCESS_GROUP", False):
            os.killpg(proc.pid, signal.SIGTERM)
        else:
            proc.terminate()
        proc.wait(timeout=10)
    except ProcessLookupError:
        pass
    except subprocess.TimeoutExpired:
        try:
            if globals().get("LLM_SERVER_PROC_USES_PROCESS_GROUP", False):
                os.killpg(proc.pid, signal.SIGKILL)
            else:
                proc.kill()
        except ProcessLookupError:
            pass
        proc.wait(timeout=10)


def resolve_model_path():
    existing = globals().get("model_path")
    if existing is not None and Path(str(existing)).exists():
        return Path(str(existing))

    model_dir = Path("/content/models")
    model_dir.mkdir(parents=True, exist_ok=True)
    local_model = model_dir / MODEL_FILE
    if local_model.exists():
        globals()["model_path"] = str(local_model)
        return local_model

    print("Resolving model path. If the model is already cached this should be quick.")
    resolved = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        local_dir=str(model_dir),
        local_dir_use_symlinks=False,
    )
    globals()["model_path"] = resolved
    return Path(resolved)


def start_llm_server(model_path):
    global LLM_SERVER_PROC, LLM_SERVER_PROC_USES_PROCESS_GROUP, LLM_SERVER_LOG_PATH, LLM_SERVER_LOG
    LLM_SERVER_LOG_PATH = Path("/content/llama_cpp_server.log")
    LLM_SERVER_LOG = open(LLM_SERVER_LOG_PATH, "w")
    llm_cmd = [
        sys.executable, "-m", "llama_cpp.server",
        "--model", str(model_path),
        "--host", "127.0.0.1",
        "--port", str(LLM_PORT),
        "--n_ctx", str(N_CTX),
        "--n_gpu_layers", str(N_GPU_LAYERS),
        "--chat_format", "chatml",
    ]
    print("Starting LLM server:", " ".join(llm_cmd))
    LLM_SERVER_PROC = subprocess.Popen(
        llm_cmd,
        stdout=LLM_SERVER_LOG,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    LLM_SERVER_PROC_USES_PROCESS_GROUP = True


def ensure_llm_server():
    ok, payload = llm_server_reachable(timeout=3)
    if ok:
        print("LLM server reachable:", payload)
        return

    print(f"LLM server is not reachable; restarting it at {llm_models_url()}.")
    print("Last connection error:", repr(payload))
    stop_llm_server()
    start_llm_server(resolve_model_path())

    for second in range(240):
        if LLM_SERVER_PROC.poll() is not None:
            LLM_SERVER_LOG.flush()
            log_tail = LLM_SERVER_LOG_PATH.read_text(errors="replace")[-4000:]
            print(log_tail)
            raise RuntimeError("LLM server exited early during automatic restart")
        ok, payload = llm_server_reachable(timeout=2)
        if ok:
            print("LLM server ready after restart:", payload)
            return
        if second % 10 == 0:
            print(f"waiting for restarted LLM server... {second}s")
        time.sleep(1)

    LLM_SERVER_LOG.flush()
    print(LLM_SERVER_LOG_PATH.read_text(errors="replace")[-4000:])
    raise TimeoutError("LLM server did not become ready after automatic restart")


ensure_llm_server()

problem_dir = create_problem_files(PROBLEM_TYPE)
run_dir = Path(RUN_ROOT) / f"{problem_label()}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir.parent.mkdir(parents=True, exist_ok=True)

print("LLM-free evaluator check for the selected problem:")
verify_problem_without_llm(problem_dir, PROBLEM_TYPE)

env = os.environ.copy()
env.update(problem_environment(PROBLEM_TYPE))
env.update({
    "OPENAI_API_KEY": "not-needed",
    "PYTHONPATH": f"/content/openevolve:{env.get('PYTHONPATH', '')}",
})

cmd = [
    sys.executable,
    "/content/openevolve/openevolve-run.py",
    str(problem_dir / "initial_program.py"),
    str(problem_dir / "evaluator.py"),
    "--config", str(problem_dir / "config_tutorial.yaml"),
    "--iterations", str(ITERATIONS),
    "--output", str(run_dir),
    "--api-base", LLM_API_BASE,
    "--primary-model", MODEL_ALIAS,
    "--log-level", "INFO",
]
print("Run directory:", run_dir)
print("Command:", " ".join(cmd))

line_queue = queue.Queue()


def reader_thread(pipe):
    for line in iter(pipe.readline, ""):
        line_queue.put(line.rstrip())
    pipe.close()


proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    start_new_session=True,
)
OPENEVOLVE_RUN_PROC = proc
OPENEVOLVE_RUN_PROC_USES_PROCESS_GROUP = True
threading.Thread(target=reader_thread, args=(proc.stdout,), daemon=True).start()

def checkpoint_render_key(run_dir):
    stable = stable_checkpoints(run_dir)
    if stable:
        number, _checkpoint_dir, info = stable[-1]
        metrics = info.get("metrics", {})
        score = metrics.get("combined_score")
        if score is None:
            if PROBLEM_TYPE == "circle_packing":
                score = metrics.get("sum_radii")
            elif PROBLEM_TYPE == "tsp":
                score = metrics.get("tour_length")
            elif PROBLEM_TYPE == "no_isosceles":
                score = metrics.get("subset_size")
        return ("stable", number, info.get("current_iteration", number), score, len(stable))

    checkpoints_dir = Path(run_dir) / "checkpoints"
    if checkpoints_dir.exists():
        checkpoint_names = tuple(sorted(path.name for path in checkpoints_dir.glob("checkpoint_*"))[-5:])
        return ("pending", checkpoint_names)
    return ("waiting",)


recent_lines = []
last_render_key = None
last_refresh = 0
try:
    while proc.poll() is None:
        while not line_queue.empty():
            recent_lines.append(line_queue.get())
            recent_lines = recent_lines[-20:]

        render_key = checkpoint_render_key(run_dir)
        state_changed = render_key != last_render_key
        if state_changed:
            clear_output(wait=True)
            print("OpenEvolve running...")
            print("Problem type:", PROBLEM_TYPE)
            print("Run directory:", run_dir)
            print("Recent output:")
            print("\n".join(recent_lines[-12:]) or "(no stdout yet; check logs after the run)")
            print()
            show_run_state(run_dir, env=env, problem_type=PROBLEM_TYPE)
            last_render_key = render_key
            last_refresh = time.time()
        elif time.time() - last_refresh >= 30:
            print("Still running; checkpoint unchanged since last render.")
            last_refresh = time.time()
        time.sleep(1)
except KeyboardInterrupt:
    _terminate_open_evolve_process(proc, label="interrupted OpenEvolve", use_process_group=True)
    raise
finally:
    if proc.poll() is None:
        _terminate_open_evolve_process(proc, label="unfinished OpenEvolve", use_process_group=True)

while not line_queue.empty():
    recent_lines.append(line_queue.get())

clear_output(wait=True)
print("OpenEvolve finished with exit code:", proc.returncode)
print("Problem type:", PROBLEM_TYPE)
print("Run directory:", run_dir)
print("Recent output:")
print("\n".join(recent_lines[-20:]) or "(no stdout captured)")
print()
show_run_state(run_dir, env=env, problem_type=PROBLEM_TYPE)

if proc.returncode != 0:
    log_files = sorted((run_dir / "logs").glob("*.log"))
    if log_files:
        print("\nLog tail:")
        print(log_files[-1].read_text(errors="replace")[-4000:])
    raise RuntimeError(f"OpenEvolve failed with exit code {proc.returncode}")


## 11. Re-render a Run Directory

In [ ]:
#@title Re-render a previous run
RUN_DIR_TO_RENDER = ""  #@param {type:"string"}
if RUN_DIR_TO_RENDER.strip():
    env = os.environ.copy()
    env.update(problem_environment(PROBLEM_TYPE))
    show_run_state(Path(RUN_DIR_TO_RENDER.strip()), env=env, problem_type=PROBLEM_TYPE)
else:
    print("Paste a run directory path, e.g. /content/openevolve_runs/circle_n16_hard_valid_sum_... or /content/openevolve_runs/tsp_n20_seed0_negative_length_... or /content/openevolve_runs/noiso_n8_size_minus_penalty_...")


## 12. Fallback Visualization

Run this even if the LLM did not start. It demonstrates why the score function matters.

In [ ]:
import os
from pathlib import Path

problem_dir = create_problem_files(PROBLEM_TYPE)
evaluator = load_evaluator(problem_dir)
base_updates = problem_environment(PROBLEM_TYPE)
os.environ.update(base_updates)

if PROBLEM_TYPE == "circle_packing":
    mode_key = "SCORE_MODE"
    examples = [
        ("hard_valid_sum", "initial_program.py"),
        ("hard_valid_sum", "hacked_program.py"),
        ("intentionally_bad_reported_sum", "hacked_program.py"),
        ("soft_penalty", "hacked_program.py"),
    ]
elif PROBLEM_TYPE == "tsp":
    mode_key = "TSP_SCORE_MODE"
    examples = [
        ("negative_length", "initial_program.py"),
        ("negative_length", "hacked_program.py"),
        ("intentionally_bad_reported_length", "hacked_program.py"),
        ("soft_penalty", "hacked_program.py"),
    ]
else:
    mode_key = "NOISO_SCORE_MODE"
    examples = [
        ("size_minus_penalty", "initial_program.py"),
        ("size_minus_penalty", "hacked_program.py"),
        ("hard_valid_size", "hacked_program.py"),
        ("intentionally_bad_reported_size", "hacked_program.py"),
    ]

for mode, program_name in examples:
    os.environ[mode_key] = mode
    env = os.environ.copy()
    program_path = problem_dir / program_name
    metrics = evaluator.evaluate(str(program_path))
    data = execute_program(program_path, env=env, problem_type=PROBLEM_TYPE)
    print(f"{program_name} / {mode}: {metrics}")
    plot_solution(data, PROBLEM_TYPE, f"{program_name} / {mode}")


## 13. Suggested Experiments

Circle packing:

1. Use `SCORE_MODE=hard_valid_sum` and watch whether `sum_radii` improves.
2. Use `SCORE_MODE=soft_penalty` and compare it with the validity-gated score.
3. Use `SCORE_MODE=intentionally_bad_reported_sum` and watch reward hacking.
4. Change `PACKING_N` from `16` to `26`; the problem gets harder and slower.

TSP:

1. Use `PROBLEM_TYPE=tsp` and `TSP_SCORE_MODE=negative_length` for the honest objective.
2. Change `TSP_SCORE_MODE` to `intentionally_bad_reported_length` to show why trusting reported metrics is unsafe.
3. Increase `TSP_N` from `20` to `30` and compare how quickly the tour improves.
4. Change `TSP_SEED` to create a different fixed city set.

No-isosceles subset:

1. Use `PROBLEM_TYPE=no_isosceles` and `NOISO_SCORE_MODE=size_minus_penalty` to watch subset size improve from a weak collinear seed.
2. Switch to `NOISO_SCORE_MODE=hard_valid_size` to show the sparse all-or-nothing version of the score.
3. Switch to `NOISO_SCORE_MODE=intentionally_bad_reported_size` to demonstrate reward hacking.
4. Change `NOISO_N` from `8` to `10`; the triple check is still fast, but the search space grows quickly.

For all problems, increase `TEMPERATURE` for more exploration, increase `MAX_TOKENS` if code extraction fails, or reduce it if generations are too slow.

The key lesson: OpenEvolve searches over code, but it follows the incentives created by your score function.
